In [6]:
%pip install pyAgrum

In [4]:
import itertools
import numpy as np
import networkx as nx
import pyagrum as gum

# enuntul nu da valoarea lui beta, asa ca o alegem noi (beta > 0)
beta = 1.0

# ---------------- a) graful si clicile ----------------

# varianta noua de pyAgrum: MarkovRandomField (in cele vechi se numea MarkovNet)
try:
    MRF = gum.MarkovRandomField
except AttributeError:
    MRF = gum.MarkovNet

mrf = MRF()

# 5 variabile binare, cu valorile 0 si 1
for i in range(1, 6):
    mrf.add(gum.LabelizedVariable(f"A{i}", f"A{i}", 2))

# muchiile din enunt (la retelele markov nu au directie)
edges = [("A1", "A2"), ("A2", "A4"), ("A3", "A4"), ("A4", "A5")]

# graful e un arbore, deci nu are triunghiuri: clicile maximale sunt chiar muchiile
# fiecare clica maximala primeste un factor (functie de potential)
def phi(a_i, a_j):
    # phi_C(a_C) = exp(beta * suma a_i*a_j), iar pentru o clica de 2 noduri
    # suma are un singur termen: exp(beta * a_i * a_j)
    return np.exp(beta * a_i * a_j)

for u, v in edges:
    f = mrf.addFactor([u, v])
    # ordinea valorilor in lista nu conteaza, pentru ca phi e simetrica in (a_i, a_j)
    f.fillWith([phi(0, 0), phi(0, 1), phi(1, 0), phi(1, 1)])

# determinam clicile maximale cu networkx, ca verificare
G = nx.Graph(edges)
cliques = [sorted(c) for c in nx.find_cliques(G)]
print("clicile maximale:", cliques)
# rezultat: [A1,A2], [A2,A4], [A3,A4], [A4,A5]

# desenam graful: in jupyter se poate folosi
#   import pyAgrum.lib.notebook as gnb; gnb.showMN(mrf)
# intr-un script il salvam ca imagine (are nevoie de graphviz instalat)
try:
    import pyagrum.lib.image as gumimage
    gumimage.export(mrf, "graf_markov.png")
except Exception as e:
    print("nu am putut exporta graful cu pyAgrum:", e)


# ---------------- b) probabilitatea comuna si starile maxime ----------------

# P(a1..a5) = (1/Z) * produsul potentialelor clicilor
# Z (constanta de normalizare) e suma produsului peste toate cele 2^5 stari
def unnormalized(a):
    a1, a2, a3, a4, a5 = a
    return phi(a1, a2) * phi(a2, a4) * phi(a3, a4) * phi(a4, a5)

states = list(itertools.product([0, 1], repeat=5))
Z = sum(unnormalized(a) for a in states)
print("Z =", Z)

# distributia comuna, pentru fiecare dintre cele 32 de stari
joint = {a: unnormalized(a) / Z for a in states}
print("suma probabilitatilor (trebuie sa fie 1):", sum(joint.values()))

# starile de probabilitate maxima (MAP)
p_max = max(joint.values())
map_states = [a for a, p in joint.items() if np.isclose(p, p_max)]
print("stari de probabilitate maxima:", map_states, "cu p =", p_max)
# pentru beta > 0 produsul a_i*a_j e maxim cand toate variabilele sunt 1,
# deci starea maxima e (1,1,1,1,1) cu probabilitatea e^(4*beta) / Z

# acelasi lucru cu pyAgrum: inmultim factorii si normalizam
p = None
for u, v in edges:
    p = mrf.factor({u, v}) if p is None else p * mrf.factor({u, v})
p = p / p.sum()
print("argmax pyAgrum:", p.argmax())


# ---------------- c) modelul Ising ----------------

# in Ising vrem sa favorizam vecinii cu aceeasi valoare (0,0 sau 1,1)
# trecem de la valorile {0, 1} la spini {-1, +1}: s = 2*a - 1
# atunci s_i * s_j = +1 daca vecinii sunt egali si -1 daca sunt diferiti
# potentialul devine phi(a_i, a_j) = exp(beta * s_i * s_j), cu beta > 0:
#   valori egale     -> e^beta  (mare)
#   valori diferite  -> e^-beta (mic)
def phi_ising(a_i, a_j):
    s_i, s_j = 2 * a_i - 1, 2 * a_j - 1
    return np.exp(beta * s_i * s_j)

def unnormalized_ising(a):
    a1, a2, a3, a4, a5 = a
    return (phi_ising(a1, a2) * phi_ising(a2, a4)
            * phi_ising(a3, a4) * phi_ising(a4, a5))

Z_ising = sum(unnormalized_ising(a) for a in states)
joint_ising = {a: unnormalized_ising(a) / Z_ising for a in states}

p_max_i = max(joint_ising.values())
map_ising = [a for a, p in joint_ising.items() if np.isclose(p, p_max_i)]
print("ising, stari maxime:", map_ising, "cu p =", p_max_i)
# aici apar doua stari maxime, toate 0 si toate 1, pentru ca modelul e simetric
# (nu favorizeaza nicio valoare anume, ci doar faptul ca vecinii sunt egali)

clicile maximale: [['A3', 'A4'], ['A2', 'A4'], ['A4', 'A5'], ['A1', 'A2']]
Z = 190.26457775349186
suma probabilitatilor (trebuie sa fie 1): 1.0000000000000002
stari de probabilitate maxima: [(1, 1, 1, 1, 1)] cu p = 0.2869590896939418
argmax pyAgrum: ([{'A4': 1, 'A5': 1, 'A3': 1, 'A2': 1, 'A1': 1}], 0.28695908969394174)
ising, stari maxime: [(0, 0, 0, 0, 0), (1, 1, 1, 1, 1)] cu p = 0.30093552954529984
